In [ ]:
!git clone  https://github.com/microsoft/LLaVA-Med

Cloning into 'LLaVA-Med'...
remote: Enumerating objects: 429, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 429 (delta 1), reused 1 (delta 1), pack-reused 418 (from 2)
Receiving objects: 100% (429/429), 77.09 MiB | 11.54 MiB/s, done.
Resolving deltas: 100% (122/122), done.


In [ ]:
!pip install -q -U transformers==4.36.2
!pip install tokenizers>=0.15.0
!pip install -q bitsandbytes==0.41.0 accelerate==0.21.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.3.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 21.8 MB/s eta 0:00:00


In [ ]:
%cd LLaVA-Med
!dir

/content/LLaVA-Med
CODE_OF_CONDUCT.md  docs	      images   llava	       README.md    SUPPORT.md
data		    download_data.sh  LICENSE  pyproject.toml  SECURITY.md


In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig
from llava.model import LlavaMistralForCausalLM
from transformers import AutoModelForCausalLM
import torch

import os
import requests
from PIL import Image
from io import BytesIO
from llava.conversation import conv_templates, SeparatorStyle
from llava.utils import disable_torch_init
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.mm_utils import tokenizer_image_token, get_model_name_from_path, KeywordsStoppingCriteria
from transformers import TextStreamer


/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [ ]:
def caption_image(image_file, prompt):
    if image_file.startswith('http') or image_file.startswith('https'):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_file).convert('RGB')
    disable_torch_init()
    conv_mode = "mistral_instruct"
    conv = conv_templates[conv_mode].copy()
    roles = conv.roles
    image_tensor = image_processor.preprocess(image, return_tensors='pt')['pixel_values'].half().cuda()
    print("image tensor length")
    print(len(image_tensor))
    print(image_tensor)
    inp = f"{roles[0]}: {prompt}"
    inp = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + inp
    print("input")
    print(inp)
    conv.append_message(conv.roles[0], inp)
    conv.append_message(conv.roles[1], None)
    raw_prompt = conv.get_prompt()
    print("raw prompt")
    print(raw_prompt)
    input_ids = tokenizer_image_token(raw_prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()
    #input_ids = tokenizer_image_token(raw_prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(model.device)

    print("input ids")
    print(input_ids)
    #decoded_input = tokenizer_image_token.decode(input_ids[0], skip_special_tokens=True)
    #print("Decoded input:", decoded_input)
    stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
    print("stop str")
    print(stop_str)
    keywords = [stop_str]
    stopping_criteria = KeywordsStoppingCriteria(keywords, tokenizer, input_ids)
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.inference_mode():
      #change temperture to allow flexibility of the model
      output_ids = model.generate(input_ids, images=image_tensor, do_sample=True, temperature=0.2,
                                  max_new_tokens=1024, streamer=streamer, use_cache=True)
    #print("output ids")
    #print(output_ids)
    print(output_ids.shape)
    #print("input ids shape:", input_ids.shape)
    #print(tokenizer.decode(output_ids[0, input_ids.shape[1]:]))

    outputs = tokenizer.decode(output_ids[0, :]).strip()
    print("outpus")
    #print(outputs)
    print("outputs printed")
    conv.messages[-1][-1] = outputs
    #output = outputs.rsplit('', 1)[0]
    output = outputs
    return image, output

In [ ]:
model_path = "microsoft/llava-med-v1.5-mistral-7b"
kwargs = {"device_map": "auto"}
kwargs['load_in_4bit'] = True
kwargs['quantization_config'] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)
#model = AutoModelForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True, **kwargs)
model = LlavaMistralForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True, **kwargs)
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/73.2k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.4.mlp.fc1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.19.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.5.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.5.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.16.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.19.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.22.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.4.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.embeddings.position_embedding.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.22.self_attn.out_proj.weight', 'model.vision_tower.

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [ ]:
vision_tower = model.get_vision_tower()
if not vision_tower.is_loaded:
  print("entered loading part")
  vision_tower.load_model()
vision_tower.to(device='cuda')
image_processor = vision_tower.image_processor

entered loading part


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:519: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(checkpoint_file, map_location=map

In [ ]:
#image, output = caption_image(f'insert link in this position', 'Insert Prompt in this position')

image, output = caption_image(f'https://raw.githubusercontent.com/microsoft/LLaVA-Med/main/llava/serve/examples/med_img_1.png', 'Tell me about the image')
print(output)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


image tensor length
1
tensor([[[[-1.0771, -1.0771, -1.0479,  ..., -1.1064, -1.0918, -1.0918],
          [-1.0771, -1.0918, -1.0479,  ..., -1.1064, -1.0771, -1.0918],
          [-1.0918, -1.0918, -1.0625,  ..., -1.1064, -1.0918, -1.0918],
          ...,
          [-0.9019, -0.7998, -0.7119,  ..., -1.1064, -1.0771, -1.0332],
          [-0.8726, -0.7705, -0.7119,  ..., -1.1064, -1.0332, -0.9312],
          [-0.8579, -0.7705, -0.6973,  ..., -1.1211, -1.0039, -0.7559]],

         [[-1.0469, -1.0469, -1.0166,  ..., -1.0771, -1.0615, -1.0469],
          [-1.0469, -1.0615, -1.0166,  ..., -1.0771, -1.0469, -1.0469],
          [-1.0615, -1.0615, -1.0312,  ..., -1.0771, -1.0615, -1.0469],
          ...,
          [-0.8364, -0.7168, -0.6265,  ..., -1.0469, -1.0166, -0.9717],
          [-0.8066, -0.7017, -0.6265,  ..., -1.0469, -0.9717, -0.8667],
          [-0.7915, -0.7017, -0.6113,  ..., -1.0615, -0.9419, -0.6865]],

         [[-0.7979, -0.7979, -0.7690,  ..., -0.8262, -0.8120, -0.7979],
        